# Gene2Wire simulation — OnDemand 0909

This notebook runs the shared, versioned paper experiment code. It contains no
dataset-specific model patches. Use a Python 3.10+ OnDemand kernel with the repository's
experiment dependencies available; the notebook does not install packages into the kernel.

Run cells from top to bottom. The first uncached use downloads the pinned code and raw
data; subsequent runs reuse verified local files. Checkpoints persist across kernel
disconnects. All result tables and predictions go beneath
`/home/yueyue/gene2wire/paper_figure_exports`; plots display here and save as PDF only.

For completed results, set `RESULTS_ONLY=True` and fill `EXISTING_EXPORT_DIRS`
with exact run directories. This skips raw-data loading and fitting, preserves the
saved scientific settings, and adds all figures and full diagnostics.
`SHOW_FULL_DIAGNOSTICS=True` is the default; set it to `False` for compact summaries.

Progress prints one summary per minute in Los Angeles local time. One unit is a
model at one repetition, fold and scenario, including selection and final refit.
All enabled baselines and controls enter the total. Detailed candidate events
remain in the exports; initial cache and final completion summaries appear once.

Outputs are deliberately cleared. Earlier paper numbers remain provisional until
the harmonized reruns and their diagnostics have been reviewed.


## Run configuration


In [ ]:
from pathlib import Path
import os

# All notebooks share these defaults. Change switches here before Run All.
N_OUTER_FOLDS = 3
USE_LOCATION = False
USE_TARGET_FEATURES = False
N_JOBS = 32
PARALLEL_UNIT = 'scenario'  # Parallelize folds × repetitions × loss/calibration settings; 'fold' also supported.
N_REPETITIONS = 5
STRATEGY = 'full_joint'
CANDIDATE_BUDGET = 32  # Per-model upper bound; keep the same setting across datasets.
SEED = 20260908

# Primary paired-reference budget. The default simulation uses this size only.
PAIRED_FRACTION = 0.20
CALIBRATION_FRACTIONS = (PAIRED_FRACTION,)  # Simulation size controls; e.g. (0.10, 0.20, 0.40).

RUN_INFORMATION_CONTROLS = True
RUN_RANDOM_FOREST = True
RUN_MECHANISM_CONTROLS = True
RUN_CALIBRATION_CONTROLS = True
RUN_QIAO = True  # Target-ID adaptation when USE_TARGET_FEATURES=False; declared descriptors when True.

# Display settings do not change scientific settings or invalidate fit checkpoints.
SHOW_PROGRESS = True
PROGRESS_LEVEL = 'summary'  # One elapsed/finished/total/Los-Angeles-time line per minute.
PROGRESS_INTERVAL_SECONDS = 60.0
SHOW_FULL_DIAGNOSTICS = True  # False switches to compact summaries.

BASE_DIR = Path('/home/yueyue/gene2wire').expanduser()
RAW_DATA_DIR = BASE_DIR / 'raw_data'
CHECKPOINT_DIR = BASE_DIR / 'checkpoints' / '0908'  # Scientific protocol cache, independent of notebook date.
EXPORT_DIR = BASE_DIR / 'paper_figure_exports'
FIGURE_DIR = BASE_DIR / 'figures' / '0909'
CODE_CACHE_DIR = BASE_DIR / 'code'

# Reload a completed export to add figures/diagnostics without fitting or raw downloads.
# Each value must be the exact run directory containing manifest.json and metrics.csv.
RESULTS_ONLY = False
EXISTING_EXPORT_DIRS = {'simulation': None}

CORE_COMMIT = '9b4c660721fd1d6347e5fac087db401fd45c7334'
EXPECTED_SOURCE_HASH = '4970ea52e7b841d4527f2f9db839067754b6ca3c4f91393370c8709778553be5'
REPO_URL = 'https://github.com/Yue-stat/Gene2Wire.git'

# Set before importing NumPy/SciPy. Only the outer fold/repetition/scenario pool is parallel.
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
                 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'

REQUIRED_MODULES = ['numpy', 'scipy', 'pandas', 'sklearn', 'joblib', 'threadpoolctl', 'matplotlib', 'yaml', 'IPython']
EXPECTED_EXPORT_LABELS = ('simulation',)


## Load the pinned code

A verified local checkout works offline.
A stale package already imported in this kernel requires a kernel restart;
the notebook never reloads or rewrites installed model source.


In [ ]:
import hashlib
import importlib.util
import re
import shutil
import subprocess
import sys
import tempfile

if sys.version_info < (3, 10):
    raise RuntimeError('Select an OnDemand Python 3.10 or newer kernel.')
if not re.fullmatch(r'[0-9a-f]{40}', CORE_COMMIT):
    raise RuntimeError('This notebook needs its released 40-character CORE_COMMIT pin.')
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_SOURCE_HASH):
    raise RuntimeError('This notebook needs its released source checksum.')

def notebook_source_hash(package_root):
    # Same byte-level convention as experiments.protocol.source_hash().
    digest = hashlib.sha256()
    for source_path in sorted(package_root.rglob('*.py')):
        digest.update(source_path.relative_to(package_root).as_posix().encode())
        digest.update(source_path.read_bytes())
    return digest.hexdigest()

def verify_checkout(checkout, require_git_pin=True):
    package_root = checkout / 'src' / 'gene2wire'
    if not package_root.is_dir():
        raise RuntimeError(f'Missing Gene2Wire sources in {checkout}')
    if require_git_pin:
        actual_commit = subprocess.check_output(
            ['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
        if actual_commit != CORE_COMMIT:
            raise RuntimeError(f'Cached code has commit {actual_commit}, expected {CORE_COMMIT}.')
    if notebook_source_hash(package_root) != EXPECTED_SOURCE_HASH:
        raise RuntimeError(f'Source checksum mismatch in {checkout}; use the released code.')
    return checkout

# Running from the exact local repository is supported without any network call.
CORE_CHECKOUT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    package_root = candidate / 'src' / 'gene2wire'
    if package_root.is_dir() and notebook_source_hash(package_root) == EXPECTED_SOURCE_HASH:
        CORE_CHECKOUT = verify_checkout(candidate, require_git_pin=False)
        break

if CORE_CHECKOUT is None:
    CODE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cached_checkout = CODE_CACHE_DIR / CORE_COMMIT
    if not cached_checkout.exists():
        stage = Path(tempfile.mkdtemp(prefix='.gene2wire-download-', dir=CODE_CACHE_DIR))
        try:
            for arguments in (
                ['git', 'init', '--quiet', str(stage)],
                ['git', '-C', str(stage), 'remote', 'add', 'origin', REPO_URL],
                ['git', '-C', str(stage), 'fetch', '--quiet', '--depth', '1', 'origin', CORE_COMMIT],
                ['git', '-C', str(stage), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'],
            ):
                subprocess.run(arguments, check=True)
            verify_checkout(stage)
            try:
                stage.rename(cached_checkout)
            except OSError:
                # Another notebook may have completed this same immutable cache.
                if not cached_checkout.exists():
                    raise
                verify_checkout(cached_checkout)
        finally:
            if stage.exists():
                shutil.rmtree(stage)
    CORE_CHECKOUT = verify_checkout(cached_checkout)

existing = sys.modules.get('gene2wire')
if existing is not None:
    same_path = Path(existing.__file__).resolve().parent == (CORE_CHECKOUT / 'src' / 'gene2wire').resolve()
    same_source = getattr(existing, '_notebook_source_hash_0908', None) == EXPECTED_SOURCE_HASH
    if not (same_path and same_source):
        raise RuntimeError('A different or unverified gene2wire is already imported. Restart the kernel, then Run All.')

missing = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Use an OnDemand Python kernel containing these dependencies: '
                       + ', '.join(missing) + '. See the repository environment instructions.')
sys.path.insert(0, str(CORE_CHECKOUT / 'src'))
import gene2wire
from gene2wire.experiments.protocol import Settings, source_hash
if source_hash() != EXPECTED_SOURCE_HASH:
    raise RuntimeError('Imported code does not match the released source checksum.')
gene2wire._notebook_source_hash_0908 = EXPECTED_SOURCE_HASH
for directory in (RAW_DATA_DIR, CHECKPOINT_DIR, EXPORT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print({'core_commit': CORE_COMMIT, 'source_hash': source_hash(),
       'imported_from': gene2wire.__file__, 'raw_cache': str(RAW_DATA_DIR),
       'checkpoints': str(CHECKPOINT_DIR), 'exports': str(EXPORT_DIR)})


## Shared scientific settings

`full_joint` evaluates simultaneous
rank/penalty candidates from a deterministic bounded Cartesian grid. Exact direct and
residual-off endpoints are included in Joint's budget. Inner validation selects models;
final preprocessing, calibration, and fitting use the designated development data.
Mechanism/calibration stress tests are simulation controls. Every model shares each
scenario's observation mask and paired-reference subset.
Qiao runs on every dataset: `USE_TARGET_FEATURES=False` uses target identity one-hot
vectors, labelled `Qiao-ID-squared` / `Qiao-ID-logit` to distinguish this adaptation.
`USE_TARGET_FEATURES=True` uses the same declared target descriptors as the other models.
No target outcomes are used to construct these descriptors.


In [ ]:
from dataclasses import asdict
import numpy as np
import pandas as pd
from IPython.display import display
from gene2wire.experiments.pipeline import run_experiment, run_simulation_experiments
from gene2wire.experiments.plotting import (
    plot_results, plot_benchmark_results, plot_information_budget_results,
    plot_detection_only_benchmark_results,
)
from gene2wire.experiments.reporting import (
    configure_compact_display, configure_full_display, display_diagnostics, load_existing_exports,
)
from gene2wire.tuning import full_joint_candidates

if SHOW_FULL_DIAGNOSTICS:
    configure_full_display()
else:
    configure_compact_display()
settings = Settings(
    n_outer_folds=N_OUTER_FOLDS, use_location=USE_LOCATION,
    use_target_features=USE_TARGET_FEATURES, n_jobs=N_JOBS, parallel_unit=PARALLEL_UNIT,
    n_repetitions=N_REPETITIONS, strategy=STRATEGY, seed=SEED,
    candidate_budget=CANDIDATE_BUDGET,
    paired_fraction=PAIRED_FRACTION, calibration_fractions=CALIBRATION_FRACTIONS,
    run_information_controls=RUN_INFORMATION_CONTROLS,
    run_random_forest=RUN_RANDOM_FOREST,
    run_mechanism_controls=RUN_MECHANISM_CONTROLS,
    run_calibration_controls=RUN_CALIBRATION_CONTROLS,
    run_qiao=RUN_QIAO,
)
if SHOW_FULL_DIAGNOSTICS:
    display(pd.DataFrame([asdict(settings)]).T.rename(columns={0: 'setting'}))
else:
    print({'folds': N_OUTER_FOLDS, 'repetitions': N_REPETITIONS, 'n_jobs': N_JOBS,
           'parallel_unit': PARALLEL_UNIT,
           'use_location': USE_LOCATION, 'use_target_features': USE_TARGET_FEATURES,
           'strategy': STRATEGY, 'run_qiao': RUN_QIAO,
           'candidate_budget': CANDIDATE_BUDGET,
           'paired_fraction': PAIRED_FRACTION, 'calibration_fractions': CALIBRATION_FRACTIONS})
if RESULTS_ONLY:
    all_artifacts = load_existing_exports(
        EXISTING_EXPORT_DIRS, expected_labels=EXPECTED_EXPORT_LABELS)
    print('RESULTS_ONLY: loaded completed exports; raw loading, preflight and fitting are skipped.')
    print('Plots and diagnostics use the SAVED manifest, not the current settings printed above.')


## Worker usage and available slots

This cell creates a single live output, refreshed once per progress interval while
the experiment runs. It reports occupied and available slots in this experiment's
scheduled worker pool, alongside requested workers and the detectable CPU allowance.
Occupied workers may be fitting, reading caches or writing outputs; this is not CPU
utilization or a count of free CPUs across other jobs. Cached work and the number of
pending tasks can reduce the scheduled pool below `N_JOBS`.


In [ ]:
from gene2wire.experiments.workers import NotebookWorkerStatus

worker_status = None
if RESULTS_ONLY:
    print('RESULTS_ONLY: no training workers are launched.')
else:
    worker_status = NotebookWorkerStatus(requested_workers=N_JOBS)


## Define input and split checks


In [ ]:
def preflight(dataset):
    dataset.validate()
    folds = tuple(dataset.split_builder(settings.n_outer_folds, settings.seed))
    for fold in folds:
        fold.validate(len(dataset.cell_ids))
    features = dataset.feature_builder(
        folds[0].train_rows, settings.use_location, settings.use_target_features)
    roles = []
    for fold in folds:
        roles.append({'dataset': dataset.name, 'outer_fold': fold.outer_fold,
                      'inner_train_cells': len(fold.train_rows),
                      'validation_cells': len(fold.validation_rows),
                      'test_cells': len(fold.test_rows),
                      'split_design': dict(fold.metadata)})
    scalar_metadata = {key: value for key, value in dataset.metadata.items()
                       if value is None or isinstance(value, (str, bool, int, float))}
    display(pd.DataFrame([{'dataset': dataset.name, 'cells': len(dataset.cell_ids),
                           'targets': len(dataset.target_ids),
                           'measured_pairs': int(dataset.measured.sum()),
                           'reference_positives': int(dataset.reference.sum()),
                           'feature_columns': features.X.shape[1],
                           'feature_blocks': dict(features.feature_blocks),
                           'target_feature_columns': 0 if features.Y_target is None else features.Y_target.shape[1],
                           'natural_paired_assay': dataset.natural_observed is not None}]))
    if SHOW_FULL_DIAGNOSTICS:
        display(pd.DataFrame(roles))
        display(pd.DataFrame([scalar_metadata]).T.rename(columns={0: 'dataset metadata'}))
    tuning = settings.tuning_config(features.X.shape[1], len(dataset.target_ids))
    display(pd.DataFrame([
        {'model': model.name, 'strategy': settings.strategy,
         'maximum_trials': tuning.candidate_budget,
         'bounded_grid_candidates': len(full_joint_candidates(model, tuning)),
         'eligible_structures': sorted({c.kind for c in full_joint_candidates(model, tuning)})}
        for model in settings.models()
    ]))
    actual_candidates = []
    for model in settings.models():
        for candidate_index, candidate in enumerate(full_joint_candidates(model, tuning), start=1):
            actual_candidates.append({
                'dataset': dataset.name, 'outer_fold': folds[0].outer_fold,
                'model': model.name, 'candidate_index': candidate_index,
                **asdict(candidate),
            })
    if SHOW_FULL_DIAGNOSTICS:
        print('Exact bounded full-joint candidates for the first inner-training fold:')
        print('For staged_rank_l2, this is grid support; realized adaptive trials are in tuning below.')
        display(pd.DataFrame(actual_candidates))
        print('Shared optimizer configuration:')
        display(pd.DataFrame([asdict(settings.fit_config())]))
    print('Features above are fitted on inner-training cells only. Final refit uses the development cells.')
    print('Repeated masks share a biological dataset; they are not additional independent animals.')
    return folds


## Simulation configuration

The three sharing strengths are run together.
`USE_LOCATION=False` excludes location from both the fitted predictor and the generated
projection signal. Setting it to `True` includes the declared location basis in both.
Generated raw arrays and truths are saved for reproducibility; truth remains outside fitting.
Each repetition generates an independent dataset; folds are aggregated within repetitions.
The run configuration defaults to `PAIRED_FRACTION=0.20` and
`CALIBRATION_FRACTIONS=(PAIRED_FRACTION,)`. All default scenarios therefore use
the same 20% paired budget. To restore the size comparison, change the latter
variable to `(0.10, 0.20, 0.40)`; existing mechanism/misspecification controls remain available.


In [ ]:
if not RESULTS_ONLY:
    from gene2wire.experiments.datasets.simulation import generate_simulation
    SHARING_STRENGTHS = (0.0, 0.5, 1.0)
    SIMULATION_OPTIONS = {
        'n_cells': 400, 'n_targets': 36, 'n_gene_features': 16,
        'n_location_features': 4, 'n_slices': 20, 'true_rank': 2,
        'n_target_features': 4, 'signal_sd': 1.35,
        'target_feature_noise': 0.85,
        'target_prevalence_range': (0.10, 0.30),
        'truth_uses_location': USE_LOCATION,
    }
    representative = generate_simulation(
        repetition=0, sharing_strength=SHARING_STRENGTHS[0], seed=SEED,
        **SIMULATION_OPTIONS)
    preflight_folds = preflight(representative)


## Run the frozen experiment matrix

This cell is the expensive step.
Existing compatible checkpoints resume automatically. Mechanism and calibration controls
follow the shared protocol; they do not multiply every dataset/rate/model combination.


In [ ]:
if not RESULTS_ONLY:
    artifacts = run_simulation_experiments(
        settings=settings, raw_cache_dir=RAW_DATA_DIR / 'simulation',
        checkpoint_dir=CHECKPOINT_DIR, export_dir=EXPORT_DIR,
        sharing_strengths=SHARING_STRENGTHS,
        simulation_options=SIMULATION_OPTIONS,
        progress=SHOW_PROGRESS, progress_level=PROGRESS_LEVEL,
        progress_interval=PROGRESS_INTERVAL_SECONDS,
        worker_status=worker_status,
    )
    all_artifacts = {'simulation': artifacts}


## Loss-rate curves and PDF figures

Curves retain the complete configured
loss-rate grid. Simulation includes all three sharing strengths; BARseq includes both panels.
The natural Projection-TAGs analysis retains its paired-label audit; no model dot panels are drawn.
No PNG files are written.


In [ ]:
for label, artifacts in all_artifacts.items():
    plot_results(artifacts, output_dir=FIGURE_DIR)


## PU-Joint and every recorded benchmark

New primary runs evaluate every retained method at 0%, 20%, 40%, 60%, and 80% loss.
Dashed lines use neither PU nor paired references; solid lines use either kind of information.
Both Prevalence methods are excluded; reference-only logistic is included.
Reference + PU logistic/MIRT/Joint use the same direct paired-reference supervision.
Older exports retain gaps where fits were not run. Single-condition model dot plots are
omitted, including natural Projection-TAGs model comparisons; its paired audit remains.
Existing primary figures above are retained. Figures display here and save as PDF.


In [ ]:
benchmark_figure_paths = {}
for label, artifacts in all_artifacts.items():
    benchmark_figure_paths[label] = plot_benchmark_results(artifacts, output_dir=FIGURE_DIR, show=True)
display(benchmark_figure_paths)


## Benchmarks without reference labels as training samples

This additional figure includes every recorded method whose projection predictor does
not train directly on paired-reference outcomes. PU logistic, PU-MIRT, PU-Joint and
sensitivity rescaling remain: paired references may calibrate their detector.
Reference-only, all Reference + PU variants, RF (paired references), and RF (observed
+ paired references) are excluded from this figure. They remain in the plots above.
Solid lines use PU or sensitivity calibration; other methods use dashed lines.
Figures display here and save as PDF. Natural and single-rate model dot panels remain off.


In [ ]:
detection_only_figure_paths = {}
for label, artifacts in all_artifacts.items():
    detection_only_figure_paths[label] = plot_detection_only_benchmark_results(
        artifacts, output_dir=FIGURE_DIR, show=True)
display(detection_only_figure_paths)


## Same paired-reference budget

Curves compare exactly Reference + PU logistic, RF (observed + paired references),
and Reference + PU-Joint. The two PU methods use reference loss on paired C and PU
loss on the remaining O. RF uses reference labels on C and raw observed labels on O,
with ordinary supervised training. Each method uses each projection outcome once;
all share the same authorized paired cells, features and splits. Other methods remain
in the all-benchmark curves above. Only recorded multi-rate comparisons are plotted;
natural and single-rate settings remain in the exported metric tables.


In [ ]:
information_budget_figure_paths = {}
for label, artifacts in all_artifacts.items():
    information_budget_figure_paths[label] = plot_information_budget_results(
        artifacts, output_dir=FIGURE_DIR, show=True)
display(information_budget_figure_paths)


## Full diagnostics and reusable exports

The report shows
endpoint metrics for every recorded method, selected-configuration frequencies, convergence,
candidate coverage and failures. Per-target metrics, every tuning trial, calibration tables,
and the saved run manifest display by default and remain in the export directory. `model_evaluation_plan.csv`
lists every unit in the progress denominator; `model_cache_accounting.csv` distinguishes
restored results, reused fits, and new/mixed fitting. An unchecked cache is not counted
as a cache miss. Progress still prints one line per minute.
`joint_selection_diagnostics.csv` reports the candidate-family counts, converged
validation minima and the selected model's margin over its own direct endpoint.
Full diagnostics can also compute this table from older exports without fitting.
Set `SHOW_FULL_DIAGNOSTICS=False` to show only the compact diagnostic summaries.


In [ ]:
# Full saved diagnostics display by default; set SHOW_FULL_DIAGNOSTICS=False for summaries.
# Trial/per-target/prediction exports also remain available on disk.
for label, artifacts in all_artifacts.items():
    display_diagnostics(artifacts, label=label, full=SHOW_FULL_DIAGNOSTICS)
print('Figure PDFs:', FIGURE_DIR)
print('Reusable raw cache:', RAW_DATA_DIR)
print('Resumable checkpoints:', CHECKPOINT_DIR)
